# 03 · Gold: star schema and KPIs

Builds the business layer as a star schema:
- **Dimensions:** `dim_date`, `dim_customer` (real customer = `customer_unique_id`), `dim_seller`, `dim_product`.
- **Facts, one grain each:** `fact_sales` (one row per item), `fact_orders` (one row per order), `fact_payments` (one row per payment).
- **KPI tables** for the dashboard: `kpi_ventas_mensuales`, `kpi_sellers`.
- **Informational PK/FK constraints**, added at the end because `CREATE OR REPLACE` drops them.

Automated checks and the GMV-to-payments reconciliation run in `05_quality_checks`.

In [0]:
# Drop foreign keys first: a dimension cannot be replaced while a fact table points to it
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")

fks = {
    "fact_sales":  ["fk_sales_customer", "fk_sales_seller", "fk_sales_product", "fk_sales_date"],
    "fact_orders": ["fk_orders_customer", "fk_orders_date"],
}
for table, names in fks.items():
    if spark.catalog.tableExists(f"workspace.gold.{table}"):
        for name in names:
            spark.sql(f"ALTER TABLE workspace.gold.{table} DROP CONSTRAINT IF EXISTS {name}")

In [0]:
%sql
-- Calendar dimension: one row per day, even days without sales
CREATE OR REPLACE TABLE workspace.gold.dim_date AS
SELECT d AS date_key, year(d) AS year, quarter(d) AS quarter, month(d) AS month,
       date_format(d, 'yyyy-MM') AS year_month, date_format(d, 'MMMM') AS month_name,
       dayofweek(d) AS day_of_week, date_format(d, 'EEEE') AS day_name,
       dayofweek(d) IN (1, 7) AS is_weekend
FROM (SELECT explode(sequence(DATE'2016-01-01', DATE'2018-12-31', INTERVAL 1 DAY)) AS d);

-- Real customer (customer_unique_id) with the latest known address
CREATE OR REPLACE TABLE workspace.gold.dim_customer AS
WITH base AS (
  SELECT c.customer_unique_id, c.zip_prefix, c.city, c.state,
         min(o.purchase_ts) OVER (PARTITION BY c.customer_unique_id) AS first_purchase_ts,
         count(*)           OVER (PARTITION BY c.customer_unique_id) AS total_orders,
         row_number()       OVER (PARTITION BY c.customer_unique_id ORDER BY o.purchase_ts DESC) AS rn
  FROM workspace.silver.customers c
  JOIN workspace.silver.orders o ON c.customer_id = o.customer_id)
SELECT b.customer_unique_id, b.zip_prefix, b.city, b.state, g.lat, g.lng,
       b.first_purchase_ts, b.total_orders, b.total_orders > 1 AS is_repeat_customer
FROM base b
LEFT JOIN workspace.silver.geolocation g ON b.zip_prefix = g.zip_prefix
WHERE b.rn = 1;

CREATE OR REPLACE TABLE workspace.gold.dim_seller AS
SELECT s.seller_id, s.zip_prefix, s.city, s.state, g.lat, g.lng
FROM workspace.silver.sellers s
LEFT JOIN workspace.silver.geolocation g ON s.zip_prefix = g.zip_prefix;

CREATE OR REPLACE TABLE workspace.gold.dim_product AS
SELECT *, length_cm * height_cm * width_cm AS volume_cm3
FROM workspace.silver.products;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Fact: one row per item sold
CREATE OR REPLACE TABLE workspace.gold.fact_sales AS
SELECT i.order_id, i.order_item_id, c.customer_unique_id, i.seller_id, i.product_id,
       CAST(o.purchase_ts AS DATE) AS date_key, o.order_status,
       i.price, i.freight_value, i.price + i.freight_value AS total_value
FROM workspace.silver.order_items i
JOIN workspace.silver.orders o    ON i.order_id = o.order_id
JOIN workspace.silver.customers c ON o.customer_id = c.customer_id;

-- Fact: one row per order (value, delivery, review)
CREATE OR REPLACE TABLE workspace.gold.fact_orders AS
WITH items AS (
  SELECT order_id, count(*) AS items_qty, count(DISTINCT seller_id) AS sellers_qty,
         sum(price) AS items_value, sum(freight_value) AS freight_value
  FROM workspace.silver.order_items GROUP BY order_id),
pay AS (
  SELECT order_id, sum(payment_value) AS paid_value, max(installments) AS max_installments,
         max_by(payment_type, payment_value) AS main_payment_type, count(*) AS payments_qty
  FROM workspace.silver.order_payments GROUP BY order_id)
SELECT o.order_id, c.customer_unique_id, CAST(o.purchase_ts AS DATE) AS date_key, o.order_status,
       i.items_qty, i.sellers_qty, i.items_value, i.freight_value,
       i.items_value + i.freight_value AS order_value,
       p.paid_value, p.main_payment_type, p.max_installments, p.payments_qty,
       datediff(o.delivered_ts, o.purchase_ts) AS delivery_days,
       o.delivery_delay_days,
       o.delivery_delay_days > 0 AS is_late,
       r.review_score,
       r.review_score <= 2 AS is_bad_review
FROM workspace.silver.orders o
JOIN workspace.silver.customers c ON o.customer_id = c.customer_id
LEFT JOIN items i ON o.order_id = i.order_id
LEFT JOIN pay p   ON o.order_id = p.order_id
LEFT JOIN workspace.silver.order_reviews r ON o.order_id = r.order_id;

-- Fact: one row per payment
CREATE OR REPLACE TABLE workspace.gold.fact_payments AS
SELECT p.order_id, p.payment_sequential, c.customer_unique_id,
       CAST(o.purchase_ts AS DATE) AS date_key,
       p.payment_type, p.installments, p.payment_value,
       round(p.payment_value / p.installments, 2) AS installment_value
FROM workspace.silver.order_payments p
JOIN workspace.silver.orders o    ON p.order_id = o.order_id
JOIN workspace.silver.customers c ON o.customer_id = c.customer_id;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- KPI tables for the dashboard (canceled and unavailable orders excluded)
CREATE OR REPLACE TABLE workspace.gold.kpi_ventas_mensuales AS
SELECT CAST(date_trunc('MONTH', date_key) AS DATE) AS month,
       count(*)                                  AS orders,
       count(DISTINCT customer_unique_id)        AS customers,
       sum(order_value)                          AS gmv,
       round(avg(order_value), 2)                AS aov,
       round(100 * avg(CAST(is_late AS INT)), 2) AS pct_late,
       round(avg(review_score), 2)               AS avg_review
FROM workspace.gold.fact_orders
WHERE order_status NOT IN ('canceled', 'unavailable')
GROUP BY 1;

CREATE OR REPLACE TABLE workspace.gold.kpi_sellers AS
SELECT s.seller_id, d.state,
       count(DISTINCT s.order_id)            AS orders,
       sum(s.total_value)                    AS gmv,
       round(avg(o.delivery_delay_days), 1)  AS avg_delay_days,
       round(avg(o.review_score), 2)         AS avg_review
FROM workspace.gold.fact_sales s
JOIN workspace.gold.fact_orders o ON s.order_id = o.order_id
JOIN workspace.gold.dim_seller d  ON s.seller_id = d.seller_id
GROUP BY 1, 2;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Informational primary and foreign keys (documentation for BI tools and Genie)
ALTER TABLE workspace.gold.dim_customer ALTER COLUMN customer_unique_id SET NOT NULL;
ALTER TABLE workspace.gold.dim_customer ADD CONSTRAINT pk_customer PRIMARY KEY (customer_unique_id);
ALTER TABLE workspace.gold.dim_seller   ALTER COLUMN seller_id SET NOT NULL;
ALTER TABLE workspace.gold.dim_seller   ADD CONSTRAINT pk_seller PRIMARY KEY (seller_id);
ALTER TABLE workspace.gold.dim_product  ALTER COLUMN product_id SET NOT NULL;
ALTER TABLE workspace.gold.dim_product  ADD CONSTRAINT pk_product PRIMARY KEY (product_id);
ALTER TABLE workspace.gold.dim_date     ALTER COLUMN date_key SET NOT NULL;
ALTER TABLE workspace.gold.dim_date     ADD CONSTRAINT pk_date PRIMARY KEY (date_key);

ALTER TABLE workspace.gold.fact_sales  ADD CONSTRAINT fk_sales_customer  FOREIGN KEY (customer_unique_id) REFERENCES workspace.gold.dim_customer;
ALTER TABLE workspace.gold.fact_sales  ADD CONSTRAINT fk_sales_seller    FOREIGN KEY (seller_id)  REFERENCES workspace.gold.dim_seller;
ALTER TABLE workspace.gold.fact_sales  ADD CONSTRAINT fk_sales_product   FOREIGN KEY (product_id) REFERENCES workspace.gold.dim_product;
ALTER TABLE workspace.gold.fact_sales  ADD CONSTRAINT fk_sales_date      FOREIGN KEY (date_key)   REFERENCES workspace.gold.dim_date;
ALTER TABLE workspace.gold.fact_orders ADD CONSTRAINT fk_orders_customer FOREIGN KEY (customer_unique_id) REFERENCES workspace.gold.dim_customer;
ALTER TABLE workspace.gold.fact_orders ADD CONSTRAINT fk_orders_date     FOREIGN KEY (date_key) REFERENCES workspace.gold.dim_date;

### Reconciliation (exploration)
The same money computed three ways. Expected: both GMVs equal (15,843,553.24) and total paid 16,008,872.12.
The gap is explained by payments on orders without items (canceled or unavailable) and installment interest.

In [0]:
%sql
SELECT
  (SELECT sum(total_value)   FROM workspace.gold.fact_sales)    AS gmv_items,
  (SELECT sum(order_value)   FROM workspace.gold.fact_orders)   AS gmv_orders,
  (SELECT sum(payment_value) FROM workspace.gold.fact_payments) AS total_paid,
  (SELECT count(*) FROM workspace.gold.fact_orders WHERE items_qty IS NULL) AS orders_without_items;

gmv_items,gmv_orders,total_paid,orders_without_items
15843553.24,15843553.24,16008872.12,775


In [0]:
%sql
-- Where does the gap come from?
SELECT
  round(sum(CASE WHEN items_qty IS NULL THEN paid_value END), 2)                  AS paid_on_orders_without_items,
  round(sum(CASE WHEN items_qty IS NOT NULL THEN paid_value - order_value END), 2) AS net_gap_on_orders_with_items,
  count_if(items_qty IS NOT NULL AND paid_value > order_value + 0.01)             AS overpaid_orders,
  count_if(paid_value IS NULL)                                                     AS orders_without_payment
FROM workspace.gold.fact_orders;

paid_on_orders_without_items,net_gap_on_orders_with_items,overpaid_orders,orders_without_payment
162591.95,2870.39,264,4


In [0]:
%sql
-- Do overpaid orders use more installments? (supports the interest hypothesis)
SELECT paid_value > order_value + 0.01 AS overpaid,
       count(*) AS orders, round(avg(max_installments), 1) AS avg_installments
FROM workspace.gold.fact_orders
WHERE items_qty IS NOT NULL AND paid_value IS NOT NULL
GROUP BY 1;

overpaid,orders,avg_installments
false,98401,2.9
true,264,6.3
